In [1]:
import numpy as np
import pandas as pd
import OpenEXR
import re
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

In [2]:
df1 = pd.read_csv("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/notrees-gt.csv")
df2 = pd.read_csv("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/trees-gt-aoi.csv")

In [3]:
df1

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.187436,0.000161,0.000196,1124.690170,1020.232214,0.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
1,0.092571,0.000277,0.000303,1124.690170,1020.232214,1.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
2,0.057824,0.000337,0.000359,1124.690170,1020.232214,2.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
3,0.171885,0.000224,0.000273,1124.690170,1020.232214,3.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
4,0.110673,0.000253,0.000287,1124.690170,1020.232214,4.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
329615,0.222720,0.001223,0.001876,262.715686,200.963755,15.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
329616,0.204124,0.001236,0.001864,262.715686,200.963755,16.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
329617,0.231891,0.001070,0.001640,262.715686,200.963755,17.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
329618,0.235853,0.000894,0.001359,262.715686,200.963755,18.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26


In [4]:
df2

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.210754,0.000179,0.000229,1136.471,1031.228,0.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
1,0.123970,0.000280,0.000321,1136.471,1031.228,1.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
2,0.092202,0.000341,0.000383,1136.471,1031.228,2.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
3,0.191730,0.000242,0.000306,1136.471,1031.228,3.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
4,0.145418,0.000262,0.000312,1136.471,1031.228,4.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
333755,0.222339,0.001229,0.001882,258.003,197.920,15.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
333756,0.204217,0.001243,0.001869,258.003,197.920,16.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
333757,0.230469,0.001081,0.001654,258.003,197.920,17.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
333758,0.234346,0.000904,0.001369,258.003,197.920,18.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26


In [5]:
big_df = pd.concat([df1, df2], ignore_index=True)
big_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.187436,0.000161,0.000196,1124.69017,1020.232214,0.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
1,0.092571,0.000277,0.000303,1124.69017,1020.232214,1.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
2,0.057824,0.000337,0.000359,1124.69017,1020.232214,2.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
3,0.171885,0.000224,0.000273,1124.69017,1020.232214,3.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
4,0.110673,0.000253,0.000287,1124.69017,1020.232214,4.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
663375,0.222339,0.001229,0.001882,258.00300,197.920000,15.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
663376,0.204217,0.001243,0.001869,258.00300,197.920000,16.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
663377,0.230469,0.001081,0.001654,258.00300,197.920000,17.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26
663378,0.234346,0.000904,0.001369,258.00300,197.920000,18.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.26


In [6]:
big_df.to_csv("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/combined-gt-aoi.csv", index=False)